# Use examples of [edges](https://github.com/romainsacchi/edges)

Author: [romainsacchi](https://github.com/romainsacchi)

This notebook shows examples on how to use `edge` to use exchange-specific
characterization factors in the characterization matrix of `bw2calc`, combining the use of exchange names and locations.

## Requirements

* **Pyhton 3.10 or higher (up to 3.11) is highly recommended**

# Use case with [brightway2](https://brightway.dev/)

`brightway2` is an open source LCA framework for Python.
To use `premise` from `brightway2`, it requires that you have an activated `brightway2` project with a `biosphere3` database as well as an [ecoinvent](https://ecoinvent.prg) v.3 cut-off or consequential database registered in that project. Please refer to the brightway [documentation](https://brightway.dev) if you do not know how to create a project and install ecoinvent.

In [ ]:
from edges import EdgeLCIA, get_available_methods, setup_package_logging
import bw2data

In [2]:
bw2data.databases

Databases dictionary with 2 object(s):
	biosphere3
	ecoinvent-3.10.1-cutoff

One can simply build its own LCIA file.
Let's consider the following LCIA file (saved under `lcia_example_2.json`):

In [3]:
{
  "name": "Example LCIA Method",
  "version": "1.0",
  "description": "Example LCIA method for greenhouse gas emissions",
  "unit": "kg CO2e",
  "exchanges": [
    {
      "supplier": {
        "name": "Carbon dioxide",
        "operator": "startswith",
        "matrix": "biosphere"
      },
      "consumer": {
        "matrix": "technosphere",
        "location": "CH"
      },
      "value": 1.0
    },
    {
      "supplier": {
        "name": "Carbon dioxide",
        "operator": "startswith",
        "matrix": "biosphere"
      },
      "consumer": {
        "matrix": "technosphere",
        "location": "FR"
      },
      "value": 2.0
    },
    {
      "supplier": {
        "name": "Carbon dioxide",
        "operator": "startswith",
        "matrix": "biosphere"
      },
      "consumer": {
        "matrix": "technosphere",
        "location": "RER"
      },
      "value": 3.0
    },
    {
      "supplier": {
        "name": "Carbon dioxide",
        "operator": "startswith",
        "matrix": "biosphere"
      },
      "consumer": {
        "matrix": "technosphere",
        "location": "CA"
      },
      "value": 15
    }
  ]
}



{'name': 'Example LCIA Method',
 'version': '1.0',
 'description': 'Example LCIA method for greenhouse gas emissions',
 'unit': 'kg CO2e',
 'exchanges': [{'supplier': {'name': 'Carbon dioxide',
    'operator': 'startswith',
    'matrix': 'biosphere'},
   'consumer': {'matrix': 'technosphere', 'location': 'CH'},
   'value': 1.0},
  {'supplier': {'name': 'Carbon dioxide',
    'operator': 'startswith',
    'matrix': 'biosphere'},
   'consumer': {'matrix': 'technosphere', 'location': 'FR'},
   'value': 2.0},
  {'supplier': {'name': 'Carbon dioxide',
    'operator': 'startswith',
    'matrix': 'biosphere'},
   'consumer': {'matrix': 'technosphere', 'location': 'RER'},
   'value': 3.0},
  {'supplier': {'name': 'Carbon dioxide',
    'operator': 'startswith',
    'matrix': 'biosphere'},
   'consumer': {'matrix': 'technosphere', 'location': 'CA'},
   'value': 15}]}

In [4]:
bw2data.databases

Databases dictionary with 2 object(s):
	biosphere3
	ecoinvent-3.10.1-cutoff

In [5]:
# activate the bw project
act = bw2data.Database("ecoinvent-3.10.1-cutoff").random()
act

'copper production, cathode, solvent extraction and electrowinning process' (kilogram, GLO, None)

In [6]:
act

'copper production, cathode, solvent extraction and electrowinning process' (kilogram, GLO, None)

In [7]:
LCA = EdgeLCIA(
    demand={act: 1},
    method=("some", "method"),
    filepath="lcia_example_2.json"
)
LCA.lci()

LCA.map_exchanges()

LCA.evaluate_cfs()
LCA.lcia()
LCA.score

0.36503066041242166

### Generate dataframe of characterization factors used

The `generate_cf_table` method generates a dataframe of the characterization factors used in the calculation. One can see the characterization factors used for each exchange in the system. For deterministic regionalized runs, `split_aggregate_consumers=True` expands weighted fallback consumer regions such as RER, GLO, RoW, or RoE into country rows.

In [8]:
df = LCA.generate_cf_table(split_aggregate_consumers=True)

In [9]:
# we can see under the "CF" column
# the characterization factors used for each exchange in the system
df

,supplier name,supplier categories,consumer name,consumer reference product,consumer location,consumer cpc,consumer ecospold01categories,consumer isic rev.4 ecoinvent,amount,CF,impact
0,"Carbon dioxide, non-fossil, resource correction","(natural resource, in air)","sulfate pulp production, from hardwood, bleached","sulfate pulp, bleached",RER,"32112: Chemical wood pulp, other than dissolvi...",None,"1701:Manufacture of pulp, paper and paperboard",7.260190e-07,3.0,2.178057e-06
1,"Carbon dioxide, non-fossil, resource correction","(natural resource, in air)",glued solid timber production,"wood chips, dry, measured as dry mass",RER,31230: Wood in chips or particles,None,1621:Manufacture of veneer sheets and wood-bas...,1.480537e-04,3.0,4.441612e-04
2,"Carbon dioxide, non-fossil, resource correction","(natural resource, in air)","planing, board, hardwood, u=10%","shavings, hardwood, loose, measured as dry mass",CH,39283: Non-agglomerated wood waste and scrap,None,1610:Sawmilling and planing of wood,1.827733e-09,1.0,1.827733e-09
3,"Carbon dioxide, non-fossil, resource correction","(natural resource, in air)","planing, board, hardwood, u=10%","sawnwood, board, hardwood, dried (u=10%), planed",CH,"31102: Wood, sawn or chipped lengthwise, slice...",None,1610:Sawmilling and planing of wood,-4.806192e-11,1.0,-4.806192e-11
4,"Carbon dioxide, non-fossil, resource correction","(natural resource, in air)","sulfate pulp production, from softwood, unblea...","sawdust, wet, measured as dry mass",RER,3928: Sawdust and wood waste and scrap,None,"1701:Manufacture of pulp, paper and paperboard",2.843601e-08,3.0,8.530802e-08
...,...,...,...,...,...,...,...,...,...,...,...
1223,"Carbon dioxide, fossil","(air, urban air close to ground)","carbon dioxide production, liquid","carbon dioxide, liquid",RER,"34210: Hydrogen, nitrogen, oxygen, carbon diox...",None,2011:Manufacture of basic chemicals,2.421657e-07,3.0,7.264970e-07
1224,"Carbon dioxide, fossil","(air, urban air close to ground)","transport, passenger car, large size, diesel, ...","transport, passenger car, large size, diesel, ...",RER,64119: Other land transportation services of p...,None,4922:Other passenger land transport,1.206713e-05,3.0,3.620138e-05
1225,"Carbon dioxide, fossil","(air, urban air close to ground)",silicon carbide production,silicon carbide,RER,34280: Hydrogen peroxide; phosphides; carbides...,None,2391:Manufacture of refractory products,1.185752e-06,3.0,3.557255e-06
1226,"Carbon dioxide, fossil","(air, urban air close to ground)","formic acid production, methyl formate route",formic acid,RER,34120: Industrial monocarboxylic fatty acids; ...,None,2011:Manufacture of basic chemicals,9.551826e-11,3.0,2.865548e-10


As expected, only CH, FR and RER-based consumers have been considered in the calculation.

In [10]:
df.groupby("consumer location")["CF"].mean()

consumer location
CA     15.0
CH      1.0
FR      2.0
RER     3.0
Name: CF, dtype: float64

But `statistics()` tells us that many exchanges have been ignored.

In [11]:
LCA.statistics()

+----------------------+---------------------------------------+
|       Activity       |  copper production, cathode, solvent  |
|                      | extraction and electrowinning process |
|     Method name      |           ('some', 'method')          |
|         Unit         |                kg CO2e                |
|      Data file       |             lcia_example_2            |
|    CFs in method     |                   4                   |
|       CFs used       |                  1228                 |
|   Unique CFs used    |                   4                   |
|  Exc. characterized  |                  1228                 |
| Exc. uncharacterized |                 334784                |
+----------------------+---------------------------------------+


We can extend the application of CFs to other exchanges, using the helping functions: `map_aggregate_locations`, `map_disaggregate_location`, `map_dynamic_locations` and `map_remaining_locations_to_global`.

* `map_aggregate_locations` maps exchanges with aggregate locations (e.g., "RER", RLA", etc.) to weighted-average CFs.
* `map_disaggregate_location` maps exchanges with locations that belong to aggregate locations ("CA-QC") to CFs of aggregate locations (e.g., "CA").
* `map_dynamic_locations` maps exchanges with dynamic locations (e.g., "RoW", "RoW") to CFs calculated as the weighted-average of all locations minus those excluded by dynamic locations' perimeter (e.g., "GLO minus CH").
* `map_remaining_locations_to_global` maps exchanges with remaining locations to the global CFs.

In [ ]:
LCA = EdgeLCIA(
    demand={act: 1},
    method=("some", "method"),
    filepath="lcia_example_2.json"
)
LCA.lci()

LCA.map_exchanges()

LCA.map_aggregate_locations()
LCA.map_dynamic_locations()
LCA.map_contained_locations()
LCA.map_remaining_locations_to_global()

LCA.evaluate_cfs()
LCA.lcia()
LCA.score

map contained locations starts
running find_locations for IE
finished find_locations
IE -> nearest is : RER
running find_locations for RNA
finished find_locations
RNA -> nearest is None
running find_locations for CN
finished find_locations
CN -> nearest is None
running find_locations for BR
finished find_locations
BR -> nearest is None
running find_locations for ID
finished find_locations
ID -> nearest is None
running find_locations for US
finished find_locations
US -> nearest is None
running find_locations for GLO
finished find_locations
GLO -> nearest is None
running find_locations for AZ
finished find_locations
AZ -> nearest is None
running find_locations for IN-HR
finished find_locations
IN-HR -> nearest is None
running find_locations for IN-MH
finished find_locations
IN-MH -> nearest is None
running find_locations for Europe without Switzerland
finished find_locations
Europe without Switzerland -> nearest is : RER
running find_locations for IN-UT
finished find_locations
IN-UT -> n

12.828075349786848

In [23]:
LCA.statistics()

+----------------------+---------------------------------------+
|       Activity       |  copper production, cathode, solvent  |
|                      | extraction and electrowinning process |
|     Method name      |           ('some', 'method')          |
|         Unit         |                kg CO2e                |
|      Data file       |             lcia_example_2            |
|    CFs in method     |                   4                   |
|       CFs used       |                  6464                 |
|   Unique CFs used    |                   6                   |
|  Exc. characterized  |                  6464                 |
| Exc. uncharacterized |                 329548                |
+----------------------+---------------------------------------+


In [24]:
LCA.weights

{('__ANY__', 'CH'): 8921981.0,
 ('__ANY__', 'FR'): 66548530.0,
 ('__ANY__', 'RER'): 0.0,
 ('__ANY__', 'CA'): 39742430.0}

In [25]:
df = LCA.generate_cf_table(split_aggregate_consumers=True)

In [26]:
df.loc[df["consumer location"]=="CA-NF"]

,supplier name,supplier categories,consumer name,consumer reference product,consumer location,consumer cpc,consumer ecospold01categories,consumer isic rev.4 ecoinvent,amount,CF,impact
2001,"Carbon dioxide, fossil","(air, urban air close to ground)","electricity production, oil","electricity, high voltage",CA-NF,17100: Electrical energy,oil/power plants,"3510:Electric power generation, transmission a...",2.998160e-05,3.0000,8.994481e-05
2002,"Carbon dioxide, fossil","(air,)","electricity production, oil","electricity, high voltage",CA-NF,17100: Electrical energy,oil/power plants,"3510:Electric power generation, transmission a...",2.368450e-07,3.0000,7.105351e-07
3946,"Carbon dioxide, from soil or biomass stock","(air, non-urban air or from high stacks)","electricity production, hydro, reservoir, non-...","electricity, high voltage",CA-NF,17100: Electrical energy,None,"3510:Electric power generation, transmission a...",2.730917e-04,1.8818,5.139040e-04


In [16]:
df.loc[df["consumer location"]=="CA-NF"]

,supplier name,supplier categories,consumer name,consumer reference product,consumer location,consumer cpc,consumer ecospold01categories,consumer isic rev.4 ecoinvent,amount,CF,impact
2415,"Carbon dioxide, fossil","(air, urban air close to ground)","electricity production, oil","electricity, high voltage",CA-NF,17100: Electrical energy,oil/power plants,"3510:Electric power generation, transmission a...",2.998160e-05,15.0,0.000450
2416,"Carbon dioxide, fossil","(air,)","electricity production, oil","electricity, high voltage",CA-NF,17100: Electrical energy,oil/power plants,"3510:Electric power generation, transmission a...",2.368450e-07,15.0,0.000004
2417,"Carbon dioxide, from soil or biomass stock","(air, non-urban air or from high stacks)","electricity production, hydro, reservoir, non-...","electricity, high voltage",CA-NF,17100: Electrical energy,None,"3510:Electric power generation, transmission a...",2.730917e-04,15.0,0.004096


We can check the weights used:

In [ ]:
LCA.weights

By default, if weights are not provided in the consumer sections of LCIA file, population is used. The other option is GDP. To be specified when calling EdgeLICA().

And we can see the weighted-average CFs used:

In [ ]:
df.groupby("consumer location")["CF"].mean()

We have pre-generated LCIA files for specific methods (AWARE, ImpactWorld+, etc.).
You can get a list like so:

In [ ]:
get_available_methods()

Check the other notebooks to see how to use them.